# Strict vs. Lax Type Coercion (Deep Dive)

As we saw in earlier modules, Pydantic defaults to **Lax Coercion**—meaning it attempts to cast incorrect data types into correct ones (e.g., converting the integer `1` into the float `1.0`). We can override this at the model level to enforce **Strict Coercion**.

## 1. Setting Global Strict Mode

We enforce strict mode using the `model_config` attribute.

```python
from pydantic import BaseModel, ConfigDict, ValidationError

class StrictModel(BaseModel):
    model_config = ConfigDict(strict=True)
    
    field_1: str
    field_2: float
    field_3: tuple

# Under Strict Mode, trying to cast a Python List to a Python Tuple FAILS
try:
    m = StrictModel(field_1="test", field_2=1.0, field_3=[1, 2, 3])
except ValidationError as e:
    print(e)
    # Output: "Input should be a valid tuple [type=tuple_type, input_value=[1, 2, 3], input_type=list]"

```

In strict mode, Pydantic refuses to cast a Python `list` into a Python `tuple`.

## 2. The JSON Exception (Why JSON Behaves Differently)

Here is where things get interesting. If you take that exact same model, in **strict mode**, and pass it a JSON payload, it *will* successfully convert the JSON Array into a Tuple!

```python
json_payload = '{"field_1": "test", "field_2": 1.0, "field_3": [1, 2, 3]}'

# This WORKS perfectly, even though it is in Strict Mode!
m_json = StrictModel.model_validate_json(json_payload)

print(type(m_json.field_3)) # <class 'tuple'>

```

**Why does this happen?**
You have to think about the native data types available in JSON. JSON only understands:

* Strings (`"hello"`)
* Numbers (`1`, `1.5`)
* Booleans (`true`, `false`)
* Null (`null`)
* Arrays (`[1, 2, 3]`)
* Objects (`{"key": "value"}`)

JSON **does not have Tuples or Sets**. It only has Arrays (`[]`). Because of this limitation, Pydantic's Strict Mode rules are deliberately relaxed when using `model_validate_json()`. If it sees a JSON Array, it is perfectly happy to coerce it into a `tuple`, `set`, or `list` because the client had no other way to transmit that data format via JSON.

However, when you use Python object instantiation (`StrictModel(field_3=[1,2,3])`), Pydantic knows you had the ability to pass a real Python tuple `(1, 2, 3)`. Because you passed a list instead, Strict Mode punishes you and throws an error.

---

## 3. Nested Type Casting with JSON

When deserializing JSON, Pydantic will deeply validate the elements inside the arrays based on your type hints.

```python
class ComplexModel(BaseModel):
    # A Tuple of arbitrary length containing only integers
    field_tuple: tuple[int, ...] 
    # A Set containing only strings
    field_set: set[str]

json_data = '{"field_tuple": [1, 2, 3], "field_set": ["a", "b", "c"]}'
complex_m = ComplexModel.model_validate_json(json_data)

print(type(complex_m.field_tuple)) # <class 'tuple'>
print(type(complex_m.field_set))   # <class 'set'>

```

If the JSON passed `{"field_set": ["a", 2, "c"]}`, Pydantic would raise a specific `ValidationError` pointing precisely to the second element (`index 1`), stating it expected a string but received an integer.

### Interview Preparation: Strict vs. Lax Coercion

<details>
<summary><b>1. You configure a Pydantic model with `ConfigDict(strict=True)`. The model expects a `tuple`. You pass a Python list directly into the class instantiation. What happens?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
Pydantic will instantly throw a `ValidationError`. Because we are in strict mode and passing native Python objects, Pydantic expects you to pass an actual Python `tuple`. It will refuse to implicitly coerce the `list` into a `tuple`.
</details>

<details>
<summary><b>2. If you take that exact same strict model from Question 1, but instead use `model_validate_json()` and pass a JSON string containing an array (e.g., `'{"my_field": [1, 2, 3]}'`), what happens? Why?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
It will succeed, and Pydantic will successfully coerce the JSON array into a Python tuple! <br><br>
This happens because JSON has a very limited set of native data types. JSON has arrays, but it does not have native constructs for tuples or sets. Pydantic is smart enough to know that a client sending JSON has no physical way to transmit a "tuple". Therefore, even in Strict Mode, Pydantic relaxes the rules specifically for JSON parsing, allowing JSON arrays to be coerced into tuples or sets.
</details>

<details>
<summary><b>3. Look at this type hint: `my_tuple: tuple[int, ...]`. What does the `...` (ellipsis) mean in this context?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
In Python type hinting, `tuple[int, ...]` signifies a tuple of <i>arbitrary length</i> where every single element must be an integer. <br><br>
If you only wrote `tuple[int]`, you are explicitly telling Pydantic (and MyPy) that you expect a tuple containing exactly one single element, which is an integer. The ellipsis tells the validator to apply the integer constraint to every element regardless of how long the tuple is.
</details>

<details>
<summary><b>4. You are consuming a JSON API. The API payload is `{"price": 10}`. Your Pydantic model is set to Strict Mode, and the field is defined as `price: float`. Will Pydantic raise an error because the JSON provided the integer `10` instead of `10.0`?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
No, it will successfully validate and coerce it to a float. <br><br>
Similar to the Array/Tuple situation, the JSON specification does not actually distinguish between integers and floats; it only defines a generic `Number` type. Therefore, when parsing JSON (even in Strict Mode), Pydantic accepts the JSON Number `10` and safely coerces it into the requested Python `float`. 
</detail